# CogMem Cognitive Patches — Minimal Experiment

**Goal:** Verify the cognitive patches architecture works.

**Plan:**
1. Load base model (4-bit, ~2GB VRAM)
2. Process first 100 tasks with N=4 candidates per task
3. Create patches from pass/fail contrasts (~20 patches expected)
4. Evaluate patches on remaining 1040 UNSEEN tasks
5. Compare: patched eval > cold eval = architecture works

**Hardware:** A4000 16GB. Model loaded in 4-bit via transformers (not Ollama).
Generation happens through model.generate() directly.


In [1]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers -q
print("Deps installed")


NVIDIA RTX A4000, 16376 MiB, 16101 MiB


Deps installed


In [2]:
!pip install "transformers==4.43.4" "sentence-transformers==2.7.0" "huggingface-hub==0.25.0" "accelerate==0.33.0" "peft==0.13.2" "bitsandbytes==0.43.3" -q


In [3]:
# Cell 2: Clone CogMem + load tasks
!cd /notebooks && git clone https://github.com/tungooxx/CogMem.git 2>/dev/null || \
    (cd /notebooks/CogMem && git pull)
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

import json
from pathlib import Path
from datasets import load_dataset

# Load BigCodeBench full (1140 tasks)
TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "entry_point": item.get("entry_point", ""),
        })
    with open(TASKS_PATH, "w") as f:
        for t in tasks:
            f.write(json.dumps(t) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

print("Tasks:", len(tasks))
# Split: first 100 for patch creation, rest for evaluation
TRAIN_TASKS = tasks[:100]
EVAL_TASKS = tasks[100:]
print("Train (create patches):", len(TRAIN_TASKS))
print("Eval (test patches):", len(EVAL_TASKS))


Already up to date.
Tasks: 1140
Train (create patches): 100
Eval (test patches): 1040


In [4]:
# Cell 3: Load base model (4-bit) + embedder
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading embedder...")
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. Free VRAM: {free:.1f} GB")
print("Ready for patch creation.")


Loading model (4-bit)...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Free VRAM: 12.8 GB
Ready for patch creation.


In [ ]:
# Cell 4: Record episodes from first 100 tasks
import time
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.memory_bank import ClusterMemoryBank
from cogmem.patches.wake import generate_with_model, find_best_contrast_pair

N_CANDIDATES = 4
MEMORY_DIR = "/notebooks/cogmem_cluster_memories"
memory_bank = ClusterMemoryBank(MEMORY_DIR)
memory_bank.load()

from peft import prepare_model_for_kbit_training
base_model = prepare_model_for_kbit_training(base_model)
print('Base model prepared for training')

start_time = time.time()
episodes_before = len(memory_bank.episodes)
total_passed = 0

for i, task in enumerate(TRAIN_TASKS):
    task_id = task["task_id"]
    prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
    task_embedding = embedder.encode(prompt).tolist()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    candidates = []
    for _ in range(N_CANDIDATES):
        try:
            response = generate_with_model(base_model, tokenizer, messages, temperature=0.8)
            code = extract_code(response)
            if code and len(code.strip()) > 20:
                result = evaluate_solution(task, code, timeout=30, mode="subprocess")
                candidates.append({"code": code, "passed": result["passed"]})
        except Exception as e:
            if i < 3:
                print('  Gen error:', type(e).__name__, str(e)[:80])

    passes = [c for c in candidates if c["passed"]]
    fails = [c for c in candidates if not c["passed"]]

    if passes:
        total_passed += 1

    if passes and fails:
        best_pair, best_sim = find_best_contrast_pair(passes, fails)
        if best_pair:
            memory_bank.record_episode(
                task_id=task_id,
                prompt=prompt,
                task_embedding=task_embedding,
                failed_code=best_pair["fail"]["code"],
                passed_code=best_pair["pass"]["code"],
                pass_fail_similarity=best_sim,
            )

    if (i + 1) % 10 == 0 or i < 5:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
        print("[{}/{}] {}: {}P/{}F | episodes={} | pass_rate={}/{} | {:.0f}/hr".format(
            i + 1, len(TRAIN_TASKS), task_id,
            len(passes), len(fails), len(memory_bank.episodes),
            total_passed, i + 1, rate))

memory_bank.save()
elapsed = (time.time() - start_time) / 60
print()
print("=" * 50)
print("EPISODE RECORDING COMPLETE")
print("Tasks processed:", len(TRAIN_TASKS))
print("Tasks with passes:", total_passed)
print("New episodes recorded:", len(memory_bank.episodes) - episodes_before)
print("Time:", round(elapsed, 1), "min")
print("Memory bank stats:", memory_bank.stats())


In [ ]:
# Cell 4b: Build cluster memories and inspect distilled artifacts
import numpy as np
import torch
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT
from cogmem.patches.compose import PatchedModel
from cogmem.patches.memory_bank import (
    ClusterMemoryBank,
    compute_applicability,
    score_memory_promotion,
    score_memory_use,
)
from cogmem.patches.wake import generate_with_model

MEMORY_DIR = "/notebooks/cogmem_cluster_memories"
memory_bank = ClusterMemoryBank(MEMORY_DIR)
memory_bank.load()
print('Loaded episodes:', len(memory_bank.episodes))

if len(memory_bank.episodes) == 0:
    raise RuntimeError(
        "No saved episodes found in /notebooks/cogmem_cluster_memories. "
        "Run Cell 4 to completion before building cluster memories."
    )

build_stats = memory_bank.build_memories(base_model, tokenizer)
retrievable = [m for m in memory_bank.memories if m.retrievable]
max_support = max((m.support_count for m in memory_bank.memories), default=0)
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)
print('Memory bank stats:', build_stats)
print('Retrievable memories:', len(retrievable))

print('=== CLUSTER MEMORY SUMMARY ===')
for memory in memory_bank.memories[:10]:
    payload = memory.retrievable_payload()
    print('Memory:', memory.memory_id)
    print('  family:', memory.family_label,
          'support:', memory.support_count,
          'promote:', round(memory.promotion_score, 3),
          'threshold:', round(memory.retrieval_threshold, 3),
          'retrievable:', memory.retrievable)
    print('  local_gain:', round(memory.local_support_gain, 4),
          'heldout_gain:', round(memory.held_out_steering_gain, 4),
          'transfer_gain:', round(memory.transfer_gain, 4),
          'transfer_rate:', round(memory.transfer_rate, 3))
    print('  recent_success:', round(memory.recent_success_rate, 3),
          'online_hurt:', round(memory.online_hurt_rate, 3),
          'utility_regression:', round(memory.utility_regression, 4),
          'redundancy:', round(memory.redundancy_penalty, 4))
    print('  neg_penalty:', round(memory.negative_steering_penalty, 4),
          'negatives:', len(memory.negative_episode_ids),
          'markers:', memory.structural_markers[:5])
    print('  payload keys:', list(payload.keys()))
    print('  patches:', payload['patch_ids'])

if not retrievable:
    print('No retrievable memories yet. Add more episodes or inspect family clustering.')
else:
    print('\n[1] Distilled artifact magnitudes:')
    for memory in retrievable[:3]:
        active = memory_bank.load_patches_for_memories([memory])
        if not active:
            print('  {}: no artifact patch loaded'.format(memory.memory_id))
            continue
        patch = active[0]
        norms = []
        for _, w in patch.lora_weights.items():
            nA = torch.norm(w['A']).item()
            nB = torch.norm(w['B']).item()
            norms.append(nA + nB)
        avg = sum(norms) / len(norms) if norms else 0.0
        first_key = list(patch.lora_weights.keys())[0]
        w = patch.lora_weights[first_key]
        print('  {} -> {}: |A|={:.4f} |B|={:.4f} avg_norm={:.4f}'.format(
            memory.memory_id[:36], patch.patch_id[:36],
            torch.norm(w['A']).item(), torch.norm(w['B']).item(), avg))
        patch.unload_weights()

    print('\n[2] Output difference (first 3 eval tasks):')
    for task in EVAL_TASKS[:3]:
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        emb = embedder.encode(prompt).tolist()
        emb_arr = np.asarray(emb, dtype=np.float32)
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]

        torch.manual_seed(42)
        out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

        active_memories, active = memory_bank.get_active_patches(emb, prompt, top_k=1, return_memories=True)

        torch.manual_seed(42)
        try:
            with PatchedModel(base_model, active, scaling_factor=0.25):
                out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)
        finally:
            for patch in active:
                patch.unload_weights()

        if not active_memories:
            print('  {}: ABSTAINED'.format(task['task_id']))
            continue

        best = active_memories[0]
        use_score = score_memory_use(best, emb_arr, prompt, max_reuse=max_reuse)
        applicability = compute_applicability(best, emb_arr, prompt)
        promote = score_memory_promotion(best, max_support=max_support)
        if out_cold == out_patched:
            print('  {}: IDENTICAL | use={:.3f} app={:.3f} promote={:.3f}'.format(
                task['task_id'], use_score, applicability, promote))
        else:
            cold_tokens = out_cold.split()
            patched_tokens = out_patched.split()
            diff = sum(1 for a, b in zip(cold_tokens, patched_tokens) if a != b)
            total = max(len(cold_tokens), len(patched_tokens), 1)
            print('  {}: {:.0f}% tokens different | memory={} | use={:.3f} | app={:.3f} | threshold={:.3f} | promote={:.3f}'.format(
                task['task_id'], diff / total * 100, best.memory_id,
                use_score, applicability, best.retrieval_threshold, promote))




In [5]:
from cogmem.patches.memory_bank import ClusterMemoryBank
memory_bank = ClusterMemoryBank("/notebooks/cogmem_cluster_memories")
memory_bank.load()
print(f"Episodes: {len(memory_bank.episodes)}")
print(f"Memories: {len(memory_bank.memories)}")
print(f"Artifact patches: {len(memory_bank.artifact_bank.patches)}")


2026-04-15 05:20:57.228834: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 05:20:57.228890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 05:20:57.229915: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 05:20:57.235578: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 05:20:58.154791: W tensorflow/compiler/tf2

Loaded 6 patches (0 promoted) from /notebooks/cogmem_cluster_memories/patch_artifacts
Episodes: 42
Memories: 6
Artifact patches: 6


In [ ]:
# Cell 4c: Check composition works with retrieved cluster memories
import numpy as np
import torch
from cogmem.patches.compose import PatchedModel
from cogmem.patches.memory_bank import compute_applicability, score_memory_promotion, score_memory_use

task = EVAL_TASKS[0]
prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
emb = embedder.encode(prompt).tolist()
emb_arr = np.asarray(emb, dtype=np.float32)
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}]
max_support = max((m.support_count for m in memory_bank.memories), default=0)
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)

active_memories, active = memory_bank.get_active_patches(emb, prompt, top_k=1, return_memories=True)
print('Selected memories:', [m.memory_id for m in active_memories])
print('Artifact patches:', [p.patch_id for p in active])
if active_memories:
    best = active_memories[0]
    print('Applicability:', round(compute_applicability(best, emb_arr, prompt), 4))
    print('Use score   :', round(score_memory_use(best, emb_arr, prompt, max_reuse=max_reuse), 4))
    print('Promote score:', round(score_memory_promotion(best, max_support=max_support), 4))
    print('Threshold   :', round(best.retrieval_threshold, 4))
    print('Payload keys:', list(best.retrievable_payload().keys()))
else:
    print('ABSTAINED: no memory cleared threshold')

print()
print('=== Greedy (temp=0) ===')
torch.manual_seed(42)
out_cold = generate_with_model(base_model, tokenizer, messages, temperature=0)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched = generate_with_model(base_model, tokenizer, messages, temperature=0)

print('Cold first 100:', out_cold[:100])
print('Patched first 100:', out_patched[:100])
print('IDENTICAL:', out_cold == out_patched)

print()
print('=== Hook Count ===')
pm = PatchedModel(base_model, active, scaling_factor=0.25)
pm.__enter__()
print('Hooks registered:', len(pm._hooks))
pm.__exit__(None, None, None)
print('Hooks removed:', len(pm._hooks) == 0)

print()
print('=== Low-temp (0.01) ===')
torch.manual_seed(42)
out_cold_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

torch.manual_seed(42)
with PatchedModel(base_model, active, scaling_factor=0.25):
    out_patched_lt = generate_with_model(base_model, tokenizer, messages, temperature=0.01)

print('IDENTICAL:', out_cold_lt == out_patched_lt)

for patch in active:
    patch.unload_weights()



In [6]:
# Cell 5: Inspect gated use scores on the first 200 unseen tasks
import numpy as np
from collections import Counter
from cogmem.patches.memory_bank import compute_applicability, score_memory_use

UNSEEN_SIM_SIZE = 200
unseen_subset = EVAL_TASKS[:UNSEEN_SIM_SIZE]
retrievable_memories = [m for m in memory_bank.memories if m.retrievable]
max_reuse = max((m.reuse_count for m in memory_bank.memories), default=0)
print("Unseen tasks inspected:", len(unseen_subset))
print("Retrievable memories:", len(retrievable_memories))

if not retrievable_memories:
    print("No retrievable memories yet.")
else:
    top1_use_scores = []
    top1_applicability = []
    score_margins = []
    abstained = 0
    memory_hits = Counter()
    family_hits = Counter()
    rows = []

    for task in unseen_subset:
        prompt = task.get("instruct_prompt", task.get("complete_prompt", ""))
        emb = np.asarray(embedder.encode(prompt).tolist(), dtype=np.float32)
        scored = []
        for memory in retrievable_memories:
            use_score = score_memory_use(memory, emb, prompt, max_reuse=max_reuse)
            applicability = compute_applicability(memory, emb, prompt)
            scored.append((use_score, applicability, memory))
        scored.sort(key=lambda item: item[0], reverse=True)
        best_score, best_applicability, best_memory = scored[0]
        passed_gate = best_score > best_memory.retrieval_threshold

        top1_use_scores.append(best_score)
        top1_applicability.append(best_applicability)
        score_margins.append(best_score - best_memory.retrieval_threshold)

        if passed_gate:
            memory_hits[best_memory.memory_id] += 1
            family_hits[best_memory.family_label] += 1
        else:
            abstained += 1

        rows.append((
            task["task_id"], best_score, best_applicability, best_memory.retrieval_threshold,
            best_memory.memory_id, best_memory.family_label, passed_gate,
        ))

    print("Mean top-1 use score: {:.3f}".format(float(np.mean(top1_use_scores))))
    print("Median top-1 use score: {:.3f}".format(float(np.median(top1_use_scores))))
    print("Mean top-1 applicability: {:.3f}".format(float(np.mean(top1_applicability))))
    print("Mean score-threshold margin: {:.3f}".format(float(np.mean(score_margins))))
    print("Abstentions: {}/{} ({:.1%})".format(abstained, len(unseen_subset), abstained / max(len(unseen_subset), 1)))

    print()
    print("Most selected memories (after gate):")
    for memory_id, hits in memory_hits.most_common():
        print("  {} -> {} tasks".format(memory_id, hits))

    print()
    print("Most selected families (after gate):")
    for family, hits in family_hits.most_common():
        print("  {} -> {} tasks".format(family, hits))

    print()
    print("Top unseen examples:")
    for task_id, score, applicability, threshold, memory_id, family, passed_gate in rows[:10]:
        print("  {} | use={:.3f} | app={:.3f} | threshold={:.3f} | {} | {} | passed={}".format(
            task_id, score, applicability, threshold, memory_id, family, passed_gate
        ))



Unseen tasks inspected: 200
Retrievable memories: 6
Mean top-1 similarity: 0.605
Median top-1 similarity: 0.624
Min/Max top-1 similarity: 0.262 / 0.818

Most matched memories:
  memory_plotting_15d1860d16 -> 97 tasks
  memory_file_io_dbbed1292e -> 39 tasks
  memory_random_numeric_bc4aaa27d3 -> 34 tasks
  memory_networking_1d3cb07294 -> 18 tasks
  memory_dataframe_50015a0265 -> 7 tasks
  memory_file_io_485b779ec5 -> 5 tasks

Matched families:
  plotting -> 97 tasks
  file_io -> 44 tasks
  random_numeric -> 34 tasks
  networking -> 18 tasks
  dataframe -> 7 tasks

Sample unseen matches:
  BigCodeBench/100 -> memory_plotting_15d1860d16 (plotting, sim=0.680)
  BigCodeBench/101 -> memory_plotting_15d1860d16 (plotting, sim=0.636)
  BigCodeBench/102 -> memory_plotting_15d1860d16 (plotting, sim=0.644)
  BigCodeBench/103 -> memory_plotting_15d1860d16 (plotting, sim=0.624)
  BigCodeBench/104 -> memory_plotting_15d1860d16 (plotting, sim=0.712)
  BigCodeBench/105 -> memory_plotting_15d1860d16 (plo

In [6]:
# Cell 5b: Sweep gated retrieval width on a small seen slice
import numpy as np
from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

DIAG_SIZE = 30
TOPK_OPTIONS = [1, 2, 5]
SCALE = 0.25

seen_subset = TRAIN_TASKS[:DIAG_SIZE]
print('Running gated retrieval sweep on', len(seen_subset), 'seen tasks')

cached = []
for i, task in enumerate(seen_subset):
    prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
    emb = embedder.encode(prompt).tolist()
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]

    cold_ok = False
    try:
        cold_response = generate_with_model(base_model, tokenizer, messages, temperature=0)
        cold_code = extract_code(cold_response)
        cold_result = evaluate_solution(task, cold_code, timeout=30, mode='subprocess')
        cold_ok = bool(cold_result['passed'])
    except Exception:
        cold_ok = False

    cached.append({
        'task': task,
        'prompt': prompt,
        'embedding': emb,
        'messages': messages,
        'cold_ok': cold_ok,
    })

    if (i + 1) % 10 == 0 or i + 1 == len(seen_subset):
        cold_passed = sum(1 for row in cached if row['cold_ok'])
        print(f'  cached cold [{i+1}/{len(seen_subset)}] cold={cold_passed}', flush=True)

results = []
cold_passed = sum(1 for row in cached if row['cold_ok'])

for top_k in TOPK_OPTIONS:
    print()
    print(f'-- top_k={top_k}, scale={SCALE} --', flush=True)
    memory_passed = 0
    helped = 0
    hurt = 0
    abstained = 0

    for i, row in enumerate(cached):
        mem_ok = row['cold_ok']
        active_memories = []
        active_patches = []

        try:
            active_memories, active_patches = memory_bank.get_active_patches(
                row['embedding'], row['prompt'], top_k=top_k, return_memories=True
            )
            if not active_patches:
                abstained += 1
            else:
                with PatchedModel(base_model, active_patches, scaling_factor=SCALE):
                    mem_response = generate_with_model(base_model, tokenizer, row['messages'], temperature=0)
                mem_code = extract_code(mem_response)
                mem_result = evaluate_solution(row['task'], mem_code, timeout=30, mode='subprocess')
                mem_ok = bool(mem_result['passed'])
        except Exception:
            mem_ok = False
        finally:
            for patch in active_patches:
                patch.unload_weights()

        if mem_ok:
            memory_passed += 1
        if (not row['cold_ok']) and mem_ok:
            helped += 1
        elif row['cold_ok'] and (not mem_ok):
            hurt += 1

        if (i + 1) % 10 == 0 or i + 1 == len(cached):
            print(f'  [{i+1}/{len(cached)}] memory={memory_passed} helped={helped} hurt={hurt} abstain={abstained}', flush=True)

    results.append({
        'top_k': top_k,
        'cold_passed': cold_passed,
        'memory_passed': memory_passed,
        'cold_rate': cold_passed / len(cached),
        'memory_rate': memory_passed / len(cached),
        'delta': (memory_passed - cold_passed) / len(cached),
        'helped': helped,
        'hurt': hurt,
        'abstained': abstained,
    })

results.sort(key=lambda r: (r['delta'], r['memory_rate'], -r['hurt']), reverse=True)
print()
print('{:<5} {:>8} {:>8} {:>8} {:>8} {:>8}'.format('k', 'cold', 'memory', 'delta', 'hurt', 'abstain'))
print('-' * 64)
for r in results:
    print('{:<5} {:>7.1%} {:>7.1%} {:>+7.1%} {:>8} {:>8}'.format(
        r['top_k'], r['cold_rate'], r['memory_rate'], r['delta'], r['hurt'], r['abstained']))
print()
print('Best setting:', results[0])


Precomputing cold results and embeddings...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  cached cold [10/30] cold=4
  cached cold [20/30] cold=5
  cached cold [30/30] cold=9
Cold baseline: 9/30 (30.0%)

[1/12] top_k=1 min_sim=0.0 margin=0.0


/usr/local/lib/python3.11/dist-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


  [10/30] memory=7 helped=3 hurt=0 abstain=0
  [20/30] memory=11 helped=6 hurt=0 abstain=1
  [30/30] memory=15 helped=6 hurt=0 abstain=8

[2/12] top_k=1 min_sim=0.0 margin=0.05
  [10/30] memory=7 helped=3 hurt=0 abstain=2
  [20/30] memory=11 helped=6 hurt=0 abstain=4
  [30/30] memory=15 helped=6 hurt=0 abstain=12

[3/12] top_k=1 min_sim=0.72 margin=0.0
  [10/30] memory=7 helped=3 hurt=0 abstain=3
  [20/30] memory=11 helped=6 hurt=0 abstain=7
  [30/30] memory=15 helped=6 hurt=0 abstain=15

[4/12] top_k=1 min_sim=0.72 margin=0.05
  [10/30] memory=7 helped=3 hurt=0 abstain=3
  [20/30] memory=11 helped=6 hurt=0 abstain=7
  [30/30] memory=15 helped=6 hurt=0 abstain=15

[5/12] top_k=2 min_sim=0.0 margin=0.0
  [10/30] memory=6 helped=2 hurt=0 abstain=0
  [20/30] memory=8 helped=4 hurt=1 abstain=1
  [30/30] memory=11 helped=4 hurt=2 abstain=8

[6/12] top_k=2 min_sim=0.0 margin=0.05
  [10/30] memory=6 helped=2 hurt=0 abstain=2
  [20/30] memory=8 helped=4 hurt=1 abstain=4
  [30/30] memory=11 hel

In [ ]:
# Cell 6: Evaluate gated cluster memories on seen and unseen tasks
import json
import logging
import traceback
from pathlib import Path

from cogmem.benchmarks.bigcodebench.prompts import SYSTEM_PROMPT, extract_code
from cogmem.benchmarks.bigcodebench.evaluator import evaluate_solution
from cogmem.patches.compose import PatchedModel
from cogmem.patches.wake import generate_with_model

EVAL_TOP_K = 1
EVAL_SCALE = 0.25
EVAL_CACHE_VERSION = 'qsplit_v2'
SEEN_EVAL_TASKS = TRAIN_TASKS
UNSEEN_EVAL_SIZE = 50
UNSEEN_EVAL_TASKS = EVAL_TASKS[:UNSEEN_EVAL_SIZE]
FORCE_RERUN_EVAL = False
EVAL_CACHE_PATH = Path('/notebooks/cogmem_cluster_memories') / (
    f'eval_cache_{EVAL_CACHE_VERSION}_seen{len(SEEN_EVAL_TASKS)}_unseen{UNSEEN_EVAL_SIZE}_'
    f'top{EVAL_TOP_K}_scale{str(EVAL_SCALE).replace('.', 'p')}.json'
)

logger = logging.getLogger('paperspace.eval')
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)

print('Seen tasks:', len(SEEN_EVAL_TASKS))
print('Unseen tasks:', len(UNSEEN_EVAL_TASKS))
print('Episodes:', len(memory_bank.episodes))
print('Memories:', len(memory_bank.memories))
print('Artifact patches:', len(memory_bank.artifact_bank.patches))
print('Eval cache:', EVAL_CACHE_PATH)

def run_eval(tasks, label):
    print()
    print('--- {} COLD + MEMORY EVAL ---'.format(label))
    cold_passed = 0
    memory_passed = 0
    abstained = 0
    used_memory = 0

    for i, task in enumerate(tasks):
        task_id = task.get('task_id', task.get('id', f'{label}_{i}'))
        prompt = task.get('instruct_prompt', task.get('complete_prompt', ''))
        task_embedding = embedder.encode(prompt).tolist()
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ]

        cold_ok = False
        mem_ok = False
        active_memories = []
        active_patches = []

        try:
            cold_response = generate_with_model(base_model, tokenizer, messages, temperature=0)
            cold_code = extract_code(cold_response)
            cold_result = evaluate_solution(task, cold_code, timeout=30, mode='subprocess')
            cold_ok = bool(cold_result['passed'])

            active_memories, active_patches = memory_bank.get_active_patches(
                task_embedding, prompt, top_k=EVAL_TOP_K, return_memories=True
            )

            if not active_patches:
                abstained += 1
                mem_ok = cold_ok
            else:
                used_memory += 1
                with PatchedModel(base_model, active_patches, scaling_factor=EVAL_SCALE):
                    response = generate_with_model(base_model, tokenizer, messages, temperature=0)
                code = extract_code(response)
                result = evaluate_solution(task, code, timeout=30, mode='subprocess')
                mem_ok = bool(result['passed'])
        except Exception as exc:
            logger.error(
                '[%s] task %s failed during get_active_patches/PatchedModel/generate_with_model/evaluate_solution: %s: %s\n%s',
                label,
                task_id,
                type(exc).__name__,
                exc,
                traceback.format_exc(limit=3).strip(),
            )
        finally:
            for patch in active_patches:
                patch.unload_weights()

        if cold_ok:
            cold_passed += 1
        if mem_ok:
            memory_passed += 1

        if (i + 1) % 25 == 0 or i + 1 == len(tasks):
            print('  [{}/{}] cold: {}/{} ({:.1%}) | memory: {}/{} ({:.1%}) | abstain={}'.format(
                i + 1, len(tasks),
                cold_passed, i + 1, cold_passed / max(i + 1, 1),
                memory_passed, i + 1, memory_passed / max(i + 1, 1),
                abstained,
            ))

    cold_rate = cold_passed / max(len(tasks), 1)
    memory_rate = memory_passed / max(len(tasks), 1)
    print('{} cold result: {} / {} ({:.1%})'.format(label, cold_passed, len(tasks), cold_rate))
    print('{} memory result: {} / {} ({:.1%})'.format(label, memory_passed, len(tasks), memory_rate))
    print('{} memory usage: used={} abstained={} ({:.1%} abstain)'.format(
        label, used_memory, abstained, abstained / max(len(tasks), 1)))
    return {
        'label': label,
        'total': len(tasks),
        'cold_passed': cold_passed,
        'cold_rate': cold_rate,
        'memory_passed': memory_passed,
        'memory_rate': memory_rate,
        'delta': memory_rate - cold_rate,
        'used_memory': used_memory,
        'abstained': abstained,
    }

if EVAL_CACHE_PATH.exists() and not FORCE_RERUN_EVAL:
    cached = json.loads(EVAL_CACHE_PATH.read_text(encoding='utf-8'))
    seen_eval = cached['seen_eval']
    unseen_eval = cached['unseen_eval']
    print('Loaded cached eval results from', EVAL_CACHE_PATH)
else:
    seen_eval = run_eval(SEEN_EVAL_TASKS, 'SEEN')
    unseen_eval = run_eval(UNSEEN_EVAL_TASKS, 'UNSEEN')
    EVAL_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    EVAL_CACHE_PATH.write_text(
        json.dumps({'seen_eval': seen_eval, 'unseen_eval': unseen_eval}, indent=2),
        encoding='utf-8',
    )
    print('Saved eval cache to', EVAL_CACHE_PATH)



Seen tasks: 100
Unseen tasks: 200
Episodes: 42
Memories: 6
Artifact patches: 6


--- SEEN MEMORY EVAL ---


KeyboardInterrupt: 

In [ ]:
# Cell 7: Results comparison + current score formulas
print('=' * 72)
print('EPISODE-FIRST CLUSTER MEMORY RESULTS')
print('=' * 72)
print()
print('Episodes recorded:', len(memory_bank.episodes))
print('Cluster memories built:', len(memory_bank.memories))
print('Artifact patches available:', len(memory_bank.artifact_bank.patches))
print('Eval top_k:', EVAL_TOP_K, '| Eval scale:', EVAL_SCALE)
print()
print('{:<12} {:>8} {:>8} {:>9} {:>9} {:>9} {:>10}'.format('Split', 'Cold', 'Memory', 'Cold %', 'Mem %', 'Delta', 'Abstain'))
print('-' * 72)
for result in [seen_eval, unseen_eval]:
    print('{:<12} {:>8} {:>8} {:>8.1%} {:>8.1%} {:>+8.1%} {:>9.1%}'.format(
        result['label'],
        result['cold_passed'],
        result['memory_passed'],
        result['cold_rate'],
        result['memory_rate'],
        result['delta'],
        result['abstained'] / max(result['total'], 1),
    ))

print()
print('Current applicability / use split:')
print('  applicability = clip(0.60 * pos_sim - 0.25 * neg_sim + 0.15 * structural_match, 0, 1)')
print('  Q_use = applicability * clip(0.45 * transfer_gain + 0.20 * recent_success_rate + 0.15 * log_reuse - 0.20 * online_hurt_rate, 0, 1)')
print('  retrieve only if Q_use > retrieval_threshold')
print()
print('Current promotion score:')
print('  Q_promote = 0.30 * heldout_gain + 0.25 * transfer_gain + 0.15 * local_support_gain')
print('             + 0.10 * distillation_success + 0.10 * log_support')
print('             - 0.15 * utility_regression - 0.15 * unseen_hurt_rate - 0.10 * redundancy_penalty')
print('  legacy q_value mirrors promotion_score for compatibility')
print('  demote if unseen_hurt_count >= 2 and unseen_hurt_count > unseen_help_count')

print()
if unseen_eval['delta'] > 0.01:
    print('Unseen-task memory improvement is positive.')
elif unseen_eval['delta'] > -0.01:
    print('Unseen-task memory effect is roughly neutral.')
else:
    print('Unseen-task memory effect is negative.')

print()
print('Memory bank:')
for key, value in memory_bank.stats().items():
    print('  {}: {}'.format(key, value))

